In [ ]:
import pandas as pd
import spacy

#Load the medium English model — includes word vectors needed for similarity and provides the tokeniser used by preprocess()
nlp = spacy.load('en_core_web_md')

#Register the SpacyTextBlob component so every processed Doc exposes doc._.blob.polarity and doc._.blob.sentiment
nlp.add_pipe('spacytextblob')


In [ ]:
#NOTE: Place the CSV file in the same directory as this notebook before running.
#Dataset: Datafiniti Amazon Consumer Reviews of Amazon Products (May 2019)
#Source: https://www.kaggle.com/datasets/datafiniti/consumer-reviews-of-amazon-products
CSV_PATH = 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv'

#Load the dataset — 28 332 rows × 24 columns of Amazon product reviews
df = pd.read_csv(CSV_PATH)

print(f"Dataset shape: {df.shape}")
df.head()


PermissionError: [Errno 13] Permission denied: 'Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products_May19.csv'

In [ ]:
# --- Preprocessing -----------------------------------------------------------

#Step 3.2.1 — Select the reviews.text column (our feature variable)
reviews_data = df['reviews.text']

#Step 3.2.2 — Drop rows where review text is missing; NaN values would cause errors in spaCy and produce meaningless sentiment scores
clean_data = df.dropna(subset=['reviews.text'])

print(f"Total rows in dataset    : {len(df)}")
print(f"Rows after removing NaN  : {len(clean_data)}")
print(f"Rows removed             : {len(df) - len(clean_data)}")


Total rows in dataset    : 28332
Rows after removing NaN  : 28332
Rows removed             : 0


In [ ]:
# --- Helper functions --------------------------------------------------------

def preprocess(text):
    """Return a cleaned, stop-word-free version of *text*.

    Steps:
    1. Cast to string, lowercase, and strip surrounding whitespace.
    2. Tokenise with spaCy.
    3. Discard stop words and whitespace tokens.
    4. Rejoin the remaining tokens into a single string.
    """
    text = str(text).lower().strip()
    doc = nlp(text)

    #is_stop flags common words ('the', 'is', 'of') that add little meaning
    filtered_tokens = [
        token.text for token in doc
        if not token.is_stop and not token.is_space
    ]

    return ' '.join(filtered_tokens)

#Step 3.3 Create Function for sentiment analysis
def analyse_sentiment(review):
    """Classify the sentiment of a single product review.

    Args:
        review (str): Raw review text.

    Returns:
        tuple: (polarity, label, sentiment)
            polarity (float) -- score from -1 (very negative) to +1 (very positive)
            label    (str)   -- 'Positive', 'Negative', or 'Neutral'
            sentiment        -- TextBlob Sentiment namedtuple (polarity, subjectivity)
    """
    cleaned_review = preprocess(review)
    doc = nlp(cleaned_review)

    polarity = doc._.blob.polarity      #strength and direction of sentiment
    sentiment = doc._.blob.sentiment    #includes subjectivity as well

    if polarity > 0:
        label = 'Positive'
    elif polarity < 0:
        label = 'Negative'
    else:
        label = 'Neutral'

    return polarity, label, sentiment


def compare_reviews(review_a, review_b):
    """Compute the semantic similarity between two review strings.

    Uses spaCy's cosine similarity over averaged word vectors.
    A score of 1.0 means the reviews are semantically identical;
    a score of 0.0 means they share no meaning.

    Args:
        review_a (str): First review text.
        review_b (str): Second review text.

    Returns:
        float: Similarity score in [0, 1].
    """
    doc_a = nlp(preprocess(review_a))
    doc_b = nlp(preprocess(review_b))
    return doc_a.similarity(doc_b)


In [ ]:
# --- Sentiment analysis on sample reviews ------------------------------------

#Step 3.4 Test on reviews spread across the dataset to check a variety of sentiments
SAMPLE_INDICES = [0, 1, 2, 10, 50]

print("=" * 70)
print("SENTIMENT ANALYSIS — SAMPLE PRODUCT REVIEWS")
print("=" * 70)

for idx in SAMPLE_INDICES:
    review_text = clean_data['reviews.text'].iloc[idx]
    polarity, label, sentiment = analyse_sentiment(review_text)

    print(f"\nReview #{idx}")
    print(f"  Text        : {review_text[:90]}...")
    print(f"  Polarity    : {polarity:+.4f}  (range: -1 very negative → +1 very positive)")
    print(f"  Subjectivity: {sentiment.subjectivity:.4f}  (0 objective → 1 subjective)")
    print(f"  Sentiment   : {label}")

print("\n" + "=" * 70)


SENTIMENT ANALYSIS — SAMPLE PRODUCT REVIEWS

Review #0
  Text        : I order 3 of them and one of the item is bad quality. Is missing backup spring so I have t...
  Polarity    : -0.4500  (range: -1 very negative → +1 very positive)
  Subjectivity: 0.3583  (0 objective → 1 subjective)
  Sentiment   : Negative

Review #1
  Text        : Bulk is always the less expensive way to go for products like these...
  Polarity    : -0.5000  (range: -1 very negative → +1 very positive)
  Subjectivity: 0.7000  (0 objective → 1 subjective)
  Sentiment   : Negative

Review #2
  Text        : Well they are not Duracell but for the price i am happy....
  Polarity    : +0.8000  (range: -1 very negative → +1 very positive)
  Subjectivity: 1.0000  (0 objective → 1 subjective)
  Sentiment   : Positive

Review #10
  Text        : I find amazon basics batteries to be equal if not superior to name brand ones. Can't belie...
  Polarity    : +0.4723  (range: -1 very negative → +1 very positive)
  Subjectivity

In [ ]:
# --- Similarity comparison between two reviews -------------------------------

#Retrieve two reviews by position; iloc prevents index-out-of-bounds errors
review_a = clean_data['reviews.text'].iloc[0]
review_b = clean_data['reviews.text'].iloc[1]

similarity_score = compare_reviews(review_a, review_b)

print("=" * 70)
print("REVIEW SIMILARITY COMPARISON")
print("=" * 70)
print(f"\nReview A : {review_a[:100]}...")
print(f"Review B : {review_b[:100]}...")
print(f"\nSimilarity score : {similarity_score:.4f}")
print("(1.0 = identical meaning | 0.0 = completely dissimilar)")
print("=" * 70)


REVIEW SIMILARITY COMPARISON

Review A : I order 3 of them and one of the item is bad quality. Is missing backup spring so I have to put a pc...
Review B : Bulk is always the less expensive way to go for products like these...

Similarity score : 0.7760
(1.0 = identical meaning | 0.0 = completely dissimilar)


# Sentiment Analysis Report

## 4.1 A Look at the Dataset

For this project, I used the Datafiniti Amazon Consumer Reviews of Amazon Products (May 2019) dataset, which I found on [Kaggle](https://www.kaggle.com/datasets/datafiniti/consumer-reviews-of-amazon-products). It's a substantial dataset containing 28,332 rows and 24 columns, covering a wide range of products including batteries, tablets, Fire TV sticks, and Kindles.

The primary feature I analysed was the `reviews.text` column, which contains the raw consumer feedback. Since the dataset was clean, I didn't need to drop any rows due to missing data.

---

## 4.2 Preprocessing Steps

To prepare the raw text for Natural Language Processing (NLP), I followed these steps:

- **Feature Selection** — I isolated the `reviews.text` column using index selection to use as my primary feature variable.
- **Data Cleaning** — I used `dropna()` to ensure the spaCy pipeline wouldn't encounter null values during processing.
- **Normalisation** — I converted all text to lowercase and stripped leading/trailing whitespace to maintain consistency for the model.
- **Tokenisation** — I used spaCy's `en_core_web_md` model to break each review down into individual linguistic units.
- **Stop-word Filtering** — I removed stop words (common terms like "the" and "of") using spaCy's `.is_stop` attribute, so the model focuses on semantically significant tokens rather than grammatical noise.

---

## 4.3 Checking the Results

I tested the sentiment analysis function on a sample of reviews to evaluate model performance. The polarity scores reflect the emotional intensity, while the labels indicate the general sentiment:

| Review | Polarity | Label |
|--------|----------|-------|
| "one of the items is bad quality..." | −0.4500 | Negative |
| "bulk is always the less expensive way..." | −0.5000 | Negative |
| "for the price I am happy" | +0.8000 | Positive |
| "equal if not superior to name brand" | +0.4723 | Positive |
| "I definitely love the price and quantity" | +0.3500 | Positive |

When comparing two battery reviews using cosine similarity, the model returned a score of **0.78**. This indicates that the model successfully identified the semantic similarity between the two reviews, recognising they discussed the same category of product.

---

## 4.4 The Good and the Not-So-Good

### What worked well

- **Unsupervised learning** — Because TextBlob relies on a pre-defined sentiment lexicon, it performed effective analysis immediately without requiring manual training or labelled examples.
- **Metric granularity** — The model provides nuanced metrics, including polarity (emotional valence) and subjectivity (factual vs. opinion-based text).
- **Computational efficiency** — The pipeline processed the large dataset quickly, demonstrating high throughput.
- **Noise reduction** — By implementing stop-word filtering, the model maintained focus on high-information tokens.

### Where there's room for improvement

- **Lack of contextual awareness** — The model struggles with sarcasm and nuanced language. For example, a neutral comment about bulk buying was incorrectly flagged as negative.
- **Stop-word risks** — Aggressive stop-word filtering can sometimes strip essential negations; for instance, "not good" could lose its negative semantic force if "not" is removed.
- **Domain sensitivity** — Since TextBlob is a general-purpose tool, it may occasionally misinterpret domain-specific technical jargon found in product reviews.
- **Label sensitivity** — The binary nature of the labelling logic (polarity > 0 or < 0) means that neutral or near-neutral sentiments are often forced into Positive or Negative categories.
